# NBA Player College Data

This is a notebook that will scrap all of the NBA players college stats

Source: [Sports Reference](https://www.sports-reference.com)

Authors: Rasheed Martin

In [1]:
import threading

import requests
import pandas as pd
import polars as pl
from bs4 import BeautifulSoup
import io

## Get Player links

To make this process simpler, we decided to get player links by using the last part of the webpage. We saw a pattern and standardization in the naming scheme so that is the most efficient method.

In [2]:
# Getting all the indexes for the players
import string
player_indexes = [f"{s}-index" for s in string.ascii_lowercase]

In [3]:
from fake_useragent import UserAgent

ua = UserAgent()

headers = {
    "User-Agent": ua.random,
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9",
    "Referer": "https://google.com",
    "Connection": "keep-alive",
}
session = requests.Session()
session.headers.update(headers)

In [13]:
# The player pages

import time
import random
import re
from pathlib import Path

for index in player_indexes:
    url = f"https://www.sports-reference.com/cbb/players/{index}.html"
    # Wait between 5 to 15 seconds randomly
    time.sleep(random.uniform(5, 15))
    
    response = session.get(url, timeout= 10)
    if response.status_code == 429:
        print("Rate limit exceeded")
        break
    soup = BeautifulSoup(response.text, "html.parser")

    # Each player link is in <p><a href="/cbb/players/{slug}.html">
    for div in soup.find_all("div"):
        paragraphs = div.find_all("p")
        for p in paragraphs:
            if p:
                player_link = p.find("a")
                note = p.find("small", class_="note")
            else:
                continue
            
            if note:
                year_string = note.get_text(strip=True)
                year_string = year_string.strip()  # Remove leading/trailing spaces
                
                # Split on dash and take the first part
                match = re.search(r"(\d{4})", year_string)
                if match and int(match.group(1)) < 1999:
                    continue
                school_link = note.find("a")
                if school_link and "women" in school_link["href"]:
                    continue
            else:
                continue
            if player_link:
                href = player_link["href"]
                if href.startswith("/cbb/players/") and href.endswith(".html"):
                    with Path("../data/raw_data/College_Stats/player_links.txt").open(mode="a") as f:
                        f.write(href + "\n")

{'/cbb/players/dewan-clayborn-1.html',
 '/cbb/players/sam-leclerc-1.html',
 '/cbb/players/aaron-creasman-1.html',
 '/cbb/players/steve-logan-1.html',
 '/cbb/players/alfredo-addesa-1.html',
 '/cbb/players/karrington-symonds-1.html',
 '/cbb/players/emil-mulic-1.html',
 '/cbb/players/ryan-jehle-1.html',
 '/cbb/players/trent-wurtz-2.html',
 '/cbb/players/cole-alexander-1.html',
 '/cbb/players/cedric-turner-1.html',
 '/cbb/players/chanse-perkins-1.html',
 '/cbb/players/donovon-wright-1.html',
 '/cbb/players/rexson-kenley-1.html',
 '/cbb/players/harlan-obioha-1.html',
 '/cbb/players/william-copeland-1.html',
 '/cbb/players/angel-alamo-1.html',
 '/cbb/players/jon-champagne-1.html',
 '/cbb/players/john-goldsberry-1.html',
 '/cbb/players/montel-james-1.html',
 '/cbb/players/andrew-scocca-1.html',
 '/cbb/players/ronnie-drinnon-1.html',
 '/cbb/players/kobe-johnson-2.html',
 '/cbb/players/malik-williams-1.html',
 '/cbb/players/alex-stewart-1.html',
 '/cbb/players/dominique-taplin-1.html',
 '/cbb/p

In [4]:
from pathlib import Path
with Path("../data/raw_data/College_Stats/player_links.txt").open(mode="r") as f:
    player_weblinks = {line.strip() for line in f}
player_weblinks

{'.html/cbb/players/kasper-suurorg-1',
 '.html/cbb/players/joey-knox-1',
 '.html/cbb/players/shevon-thompson-1',
 '.html/cbb/players/jameson-richardson-1',
 '.html/cbb/players/corey-butler-1',
 '.html/cbb/players/robert-wright-3',
 '.html/cbb/players/richard-lee-1',
 '.html/cbb/players/prosper-obidiebube-1',
 '.html/cbb/players/phil-gillette-1',
 '.html/cbb/players/tony-hickman-1',
 '.html/cbb/players/jasen-baranowski-1',
 '.html/cbb/players/james-hulbin-1',
 '.html/cbb/players/andy-roethle-1',
 '.html/cbb/players/john-burrell-1',
 '.html/cbb/players/justin-wilkerson-1',
 '.html/cbb/players/isaac-aguiar-1',
 '.html/cbb/players/aidan-saunders-1',
 '.html/cbb/players/melvin-dews-1',
 '.html/cbb/players/nic-lane-1',
 '.html/cbb/players/jules-bernard-1',
 '.html/cbb/players/royal-ivey-1',
 '.html/cbb/players/tim-lang-1',
 '.html/cbb/players/dietrich-cole-1',
 '.html/cbb/players/nathaniel-parker-1',
 '.html/cbb/players/kenny-kelly-1',
 '.html/cbb/players/will-bogan-1',
 '.html/cbb/players/j

In [5]:
len(player_weblinks)

47214

In [6]:
import logging
from pathlib import Path
from datetime import datetime


dirpath = Path("../logs")
dirpath.mkdir(parents=True, exist_ok=True)
datetime_str = datetime.now().strftime("%Y%m%d-%H")

logging.basicConfig(
    filename=str(Path(f"../logs/web_scraper_{datetime_str}.log")),
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)


In [7]:
import re
def slug_to_name(slug):
    name = re.sub(r"-\d+$", "", slug)
    return name.replace("-", " ").title()

In [8]:
len(player_weblinks)

47214

In [9]:
og_player_weblinks = player_weblinks.copy()

In [10]:
player_weblinks: set = set(player_weblinks)

In [10]:
player_weblinks = og_player_weblinks

In [11]:
import asyncio
import random
import logging
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from tqdm.asyncio import tqdm_asyncio
from utils.utils import helper
from fake_useragent import UserAgent

# Config
DELAY_EVERY = 1000
MAX_WORKERS = 4
DELAY_PER_THREAD = 1

processed_count = 0
processed_weblinks = set()
count_lock = asyncio.Lock()
rate_limit_event = asyncio.Event()
rate_limit_event.set()

async def helper_async(table, drafted, weight, height, data_type, player):
    await asyncio.to_thread(helper, table, drafted, weight, height, data_type, player)

async def countdown_timer(seconds):
    for remaining in range(seconds, 0, -1):
        if remaining % 60 == 0:
            logging.info(f"[GLOBAL] Sleeping for {remaining//60} minutes more...")
        await asyncio.sleep(1)

async def scrape_players(context, player, index):
    global processed_count

    stagger = (index % MAX_WORKERS) * DELAY_PER_THREAD
    logging.info(f"[{player}] Initial stagger: {stagger:.1f}s")
    await asyncio.sleep(stagger)

    url = f"https://www.sports-reference.com/{player}"

    while True:
        await rate_limit_event.wait()
        logging.info(f"Processing {player}...")

        try:
            await asyncio.sleep(random.uniform(5, 30))

            page = await context.new_page()
            try:
                await page.set_extra_http_headers({
                    "Accept-Language": "en-US,en;q=0.9",
                    "Referer": "https://www.google.com/"
                })

                response = await page.goto(url)
                status = response.status
                message = response.status_text
                html = await page.content()

            finally:
                await page.close()

            if status == 429:
                async with count_lock:
                    if rate_limit_event.is_set():
                        logging.warning(f"\n[{player}] Got 429! Triggering global rate limit.")
                        rate_limit_event.clear()
                        logging.info("\n[GLOBAL] Sleeping for 30 minutes due to rate limit.")
                        await countdown_timer(30 * 60)
                        logging.info("\n[GLOBAL] Finished sleeping. Unblocking tasks.")
                        rate_limit_event.set()
                continue

            elif status != 200:
                logging.error(f"\nFailed with status {status}: Response message: {message}, skipping {player}.")
                break

            soup = BeautifulSoup(html, "html.parser")
            meta_div = soup.find("div", id="meta")
            drafted = any("draft" in a.get("href", "") for a in meta_div.find_all("a"))
            weight, height = None, None
            for p in meta_div.find_all("p"):
                span = [span.text for span in p.find_all("span")]
                if span:
                    height = span[0]
                    weight = span[1] if len(span) > 1 else None

            bottom_nav = soup.find("div", id="bottom_nav_container")
            nba_link_text = None
            if bottom_nav:
                for link in bottom_nav.find_all("a"):
                    if link.get_text(strip=True) == "Basketball-Reference.com":
                        nba_link_text = link.get_text(strip=True)
                        break
            if nba_link_text is None:
                raise ValueError("Did not play in NBA")

            for table_id, label in [
                ("players_per_game", "Per_Game"),
                ("players_totals", "Total_Stats"),
                ("players_advanced", "Advanced_Stats")
            ]:
                table = soup.find("table", id=table_id)
                await helper_async(table, drafted, weight, height, data_type=label, player=player)

            print(f"\nSuccessfully parsed: {player}")

        except ValueError as e:
            logging.error(f"\nError: {e}. Skipping: {player}")
            processed_weblinks.discard(player)
            break
        except Exception as e:
            logging.error(f"\nUnexpected error: {e}. Retrying {player}...")
            continue


    async with count_lock:
        processed_count += 1
        processed_weblinks.add(player)
        if processed_count % DELAY_EVERY == 0:
            extra_sleep = random.uniform(60, 180)
            logging.info(f"[{processed_count}] Taking a long break: {extra_sleep:.1f}s")
            await asyncio.sleep(extra_sleep)

async def main():
    global player_weblinks  # Make sure this is defined somewhere
    temp_weblinks = player_weblinks - processed_weblinks
    if not temp_weblinks:
        print("No more weblinks available or finished scraping :)")
        return

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        semaphore = asyncio.Semaphore(MAX_WORKERS)

        async def sem_scrape(player, index):
            async with semaphore:
                context = await browser.new_context()
                try:
                    await scrape_players(context, player, index)
                finally:
                    await context.close()

        tasks = [asyncio.create_task(sem_scrape(player, i)) for i, player in enumerate(temp_weblinks)]
        for f in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
            await f

        await browser.close()


In [12]:
# To run in script or Jupyter, do:
await main()

  0%|          | 4/47214 [00:33<109:41:32,  8.36s/it]


CancelledError: 

# OLD CODE for Master List

In [ ]:
def build_df(list_df):
    master_df = pl.concat(list_df, how="diagonal_relaxed")
    master_df = master_df.with_columns(
        pl.col("Name").map_elements(slug_to_name, return_dtype=pl.String)
    )
    return master_df

In [ ]:
dirpath = Path("../data/raw_data/College_Stats")
dirpath.mkdir(parents=True, exist_ok=True)

df = build_df(per_df_list)
filepath = Path(dirpath / "master_list_per_game.csv")
df.write_csv(filepath, index=False)

df = build_df(total_df_list)
filepath = Path(dirpath / "master_list_total.csv")
df.write_csv(filepath, index=False)

df = build_df(advanced_df_list)
filepath = Path(dirpath / "master_list_advanced.csv")
df.write_csv(filepath, index=False)

In [119]:
master_df = master_df.drop(["Unnamed: 14"])

In [128]:
master_df.filter((pl.col("Pos").is_null()) & (pl.col("Season") == '2000-01')).unique()

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
"""2000-01""",null,36.0,2.0,22.9,3.5,8.0,0.438,0.3,1.8,3.2,6.3,0.511,0.457,2.4,4.3,0.568,6.0,1.5,1.2,1.0,2.4,2.4,9.8,"""Gerald Wallace""","""Drafted""",0.175,2.3,3.7


In [130]:
master_df.filter(pl.col("Name") == "Gerald Wallace")

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
"""2000-01""",null,36.0,2.0,22.9,3.5,8.0,0.438,0.3,1.8,3.2,6.3,0.511,0.457,2.4,4.3,0.568,6.0,1.5,1.2,1.0,2.4,2.4,9.8,"""Gerald Wallace""","""Drafted""",0.175,2.3,3.7


In [134]:
# Fixing Gerald Wallace
master_df = master_df.with_columns(
    pl.when(pl.col("Name") == "Gerald Wallace")
    .then(pl.lit("SF"))
    .otherwise(pl.col("Pos"))
    .alias("Pos")
)
master_df.filter(pl.col("Name") == "Gerald Wallace")

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
"""2000-01""","""SF""",36.0,2.0,22.9,3.5,8.0,0.438,0.3,1.8,3.2,6.3,0.511,0.457,2.4,4.3,0.568,6.0,1.5,1.2,1.0,2.4,2.4,9.8,"""Gerald Wallace""","""Drafted""",0.175,2.3,3.7


In [135]:
# Dropping Null Pos
master_df = master_df.drop_nulls(["Pos"])

In [136]:
master_df.null_count()

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,5,929,688,9,42,41,1243,1244,1243,1244,1243,1242,9,17,16,103,478,792,791,837,54,4,0,0,1387,1935,1935


In [138]:
master_df = master_df.with_columns(
    pl.col("Season").str.slice(0, 4).cast(pl.Int32).alias("Year")
)

In [145]:
# Filtering for above 1999
master_df = master_df.filter(pl.col("Year") >= 2000).sort(pl.col("Year"), descending=True)

In [147]:
master_df.select(pl.col("Year").value_counts())

Year
struct[2]
"{2021,1}"
"{2018,1}"
"{2017,2}"
"{2015,16}"
"{2014,21}"
…
"{2004,47}"
"{2003,56}"
"{2002,51}"


In [148]:

dirpath = Path("../data/raw_data/College_Stats")
dirpath.mkdir(parents=True, exist_ok=True)
filepath = Path(dirpath / "master_list_updated_above_2000.csv")
if not filepath.exists():
    master_df.write_csv(filepath, separator=",")